# 06 — Waiting-Time Model

**Layer:** Machine learning  
**Objective:** Predict next month's share of pathways seen within 18 weeks (`target_pct_within_18_next`) per provider × specialty.  
**Input:** `data/processed/rtt_features.parquet`.  
**Model:** XGBoost regressor (level and delta framings).  
**Baselines:** persistence (next = current) and drift (next = current + last change).

**Metrics:** MAE and RMSE in **percentage points** (the target is a 0–1 share). MAE measures typical error; RMSE penalises large misses more heavily.

**Evaluation:** time-based split (train before Dec 2025; test Dec 2025–Feb 2026, predicting Jan–Mar 2026).

**Execution:** select the `Python (HCIP)` kernel, then run all cells in order.

## 1. Load, features, targets

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor


def resolve_dir(*parts: str) -> Path:
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base.joinpath(*parts)
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not locate '{Path(*parts)}'.")


FEATURES_PATH = resolve_dir("data", "processed") / "rtt_features.parquet"
TEST_FROM = pd.Timestamp("2025-12-01")

FEATURE_COLS = [
    "total_waiting", "pct_within_18wk", "breach_rate", "over_52_share", "over_104_share",
    "month", "quarter", "month_sin", "month_cos",
    "lag1_total", "lag2_total", "lag3_total", "lag12_total", "lag1_breach", "lag12_breach",
    "roll3_total", "roll6_total", "roll12_total", "roll3_std_total",
    "mom_change_total", "mom_pct_total", "yoy_pct_total",
]

df = pd.read_parquet(FEATURES_PATH).dropna(
    subset=["target_pct_within_18_next", "pct_within_18wk", "lag1_breach"]
).reset_index(drop=True)
df["prev_pct"] = 1 - df["lag1_breach"]          # previous month's pct_within_18
df["target_delta_pct"] = df["target_pct_within_18_next"] - df["pct_within_18wk"]
print(f"Rows: {len(df):,}")

Rows: 77,379


## 2. Time-based split

In [2]:
train = df[df["period_date"] < TEST_FROM]
test = df[df["period_date"] >= TEST_FROM]
y_test = test["target_pct_within_18_next"].to_numpy()
current = test["pct_within_18wk"].to_numpy()
print(f"Train: {len(train):,}  Test: {len(test):,}")

Train: 66,859  Test: 10,520


## 3. Baselines and metric helper

In [3]:
def evaluate(name: str, y_true, y_pred) -> dict:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return {"model": name,
            "MAE_pp": mean_absolute_error(y_true, y_pred) * 100,
            "RMSE_pp": mean_squared_error(y_true, y_pred) ** 0.5 * 100}


results = [
    evaluate("Persistence", y_test, current),
    evaluate("Drift", y_test, current + (current - test["prev_pct"].to_numpy())),
]
pd.DataFrame(results)

,model,MAE_pp,RMSE_pp
0,Persistence,4.142747,9.045084
1,Drift,5.459006,12.777934


## 4. XGBoost (level and delta framings)

Predictions are clipped to the valid 0–1 range.

In [4]:
def make_model() -> XGBRegressor:
    return XGBRegressor(n_estimators=500, max_depth=6, learning_rate=0.05,
                        subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1)


model_level = make_model().fit(train[FEATURE_COLS], train["target_pct_within_18_next"])
pred_level = np.clip(model_level.predict(test[FEATURE_COLS]), 0, 1)
results.append(evaluate("XGBoost level", y_test, pred_level))

model_delta = make_model().fit(train[FEATURE_COLS], train["target_delta_pct"])
pred_delta = np.clip(current + model_delta.predict(test[FEATURE_COLS]), 0, 1)
results.append(evaluate("XGBoost delta", y_test, pred_delta))
print("Trained.")

Trained.


## 5. Results

Lower is better. Note MAE and RMSE can disagree: a model can have slightly higher typical error (MAE) but fewer large misses (RMSE).

In [5]:
scoreboard = pd.DataFrame(results).set_index("model").round(3)
base = scoreboard.loc["Persistence", "MAE_pp"]
scoreboard["MAE_vs_persistence_%"] = ((1 - scoreboard["MAE_pp"] / base) * 100).round(1)
scoreboard

,MAE_pp,RMSE_pp,MAE_vs_persistence_%
model,,,
Persistence,4.143,9.045,0.0
Drift,5.459,12.778,-31.8
XGBoost level,4.270,8.720,-3.1
XGBoost delta,4.200,8.553,-1.4


## 6. Feature importance (delta model)

In [6]:
pd.Series(model_delta.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False).head(12)

pct_within_18wk    0.060581
breach_rate        0.058688
roll12_total       0.054194
roll3_std_total    0.052622
lag1_breach        0.052218
lag2_total         0.049713
lag3_total         0.048929
lag12_breach       0.048584
yoy_pct_total      0.048171
roll6_total        0.046963
month_sin          0.044738
month_cos          0.044450
dtype: float32

## 7. Findings

**Result (test: Jan–Mar 2026, percentage points):**

| Model | MAE | RMSE | MAE vs persistence |
|---|---|---|---|
| Persistence | 4.14 | 9.05 | baseline |
| Drift | 5.46 | 12.78 | −32% |
| XGBoost level | 4.27 | 8.72 | −3.1% |
| XGBoost delta | 4.20 | 8.55 | −1.4% |

**Interpretation.** Like the waiting-list *level*, the share-within-18-weeks is highly persistent month to month, so persistence is again a strong baseline. XGBoost is marginally *worse* on MAE (typical error) but clearly *better* on RMSE (8.55 vs 9.05, ~5% lower) — it trades a hair of average accuracy for materially fewer large misses, which is valuable when a sudden deterioration is the costly case to miss.

**Conclusion.** This is the same regime as demand (Notebook 04): a near-persistent target where ML offers, at best, a marginal edge. The recommended deployment is the same **Forecast Value Added / champion–challenger** wrapper as the demand model — default to persistence, use the delta model only where it has demonstrably reduced error. Drift is clearly rejected (worse on both metrics).

**Project-level picture across the three models:**
- **Demand (level):** persistence-dominated — ML ≈ parity. (Nb04)
- **Waiting time (share):** persistence-dominated — ML ≈ parity on MAE, better on RMSE. (this notebook)
- **Breach (yes/no):** strong learnable signal — ML clearly wins (ROC AUC 0.978 vs 0.891). (Nb05)

The honest takeaway: the *levels* of these slow-moving series are hard to beat at a one-month horizon, but the *binary breach event* is well predicted. Baselining every model is what made that distinction visible.